In [2]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [3]:
# experimental data
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_19368/4196082941.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_19368/4196082941.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [4]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [5]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [6]:
def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):
    # Garante que q_val seja array 1D
    q_val = np.atleast_1d(q_val)
    results = []

    for q in q_val:
        def integrand(y, x, mg, a1, a2, m2_func, q_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            result = k * (
                T_1(k, q_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q_val, phi, mg, a1, a2, m2_func)
            ) * jacobian
            return result

        def inner_integral(x):
            integral_real = fixed_quad(
                lambda y: np.real(integrand(y, x, mg, a1, a2, m2_func, q)),
                0, 1, n=n_points
            )[0]
            integral_imag = fixed_quad(
                lambda y: np.imag(integrand(y, x, mg, a1, a2, m2_func, q)),
                0, 1, n=n_points
            )[0]
            return integral_real + 1j * integral_imag

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]

In [7]:
# def model function
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [8]:
# set cost and minimize
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='pl')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='pl')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)


chi2_total = chi2_7 + chi2_8 + chi2_13


minuit_born = Minuit(
    chi2_total,
    mg = 0.421,
    a1 = 1.517,
    a2 = 2.05,
    eps = 0.0753
)

minuit_born.migrad()
minuit_born.hesse()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 25.93 (χ²/ndof = 0.2)      │              Nfcn = 341              │
│ EDM = 3.1e-06 (Goal: 0.0002)     │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ eps  │  0.0616   │  0.0022   │            │            │         │         │       │
│ 1 │ mg   │   0.389   │   0.005   │            │            │         │         │       │
│ 2 │ a1   │   1.49    │   0.05    │            │            │         │         │       │
│ 3 │ a2   │   2.16    │   0.31    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │      eps       mg       a1       a2 │
├─────┼─────────────────────────────────────┤
│ eps │ 4.86e-06    10e-6    54e-6  -158e-6 │
│  mg │    10e-6 2.41e-05 0.056e-3 0.100e-3 │
│  a1 │    54e-6 0.056e-3   0.0023  -0.0136 │
│  a2 │  -158e-6 0.100e-3  -0.0136   0.0956 │
└─────┴─────────────────────────────────────┘

In [9]:
# Calculates and plot dif sigma 
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2

        integral_value = full_int(mg, a1, a2, mg_model, q2, sqrt_s)


        diff_T = integral_value
        print(diff_T)
        # print(f"q2: {q2}, diff_T: {diff_T}")

        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        dif_sigma  = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    minuit_born.values['eps'],
    minuit_born.values['mg'], 
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


(10.017599690932732+0j)
(9.96437667225915+0j)
(9.911395674964236+0j)
(9.85865632107089+0j)
(9.806158235904883+0j)
(9.753901050748969+0j)
(9.701884400474915+0j)
(9.650107921846175+0j)
(9.598571259788276+0j)
(9.547274057732402+0j)
(9.496215962013624+0j)
(9.4453966273365+0j)
(9.394815705667387+0j)
(9.344472847041073+0j)
(9.294367711583911+0j)
(9.244499960154803+0j)
(9.194869248462624+0j)
(9.145475229865609+0j)
(9.096317568707882+0j)
(9.047395929080242+0j)
(8.998709968780592+0j)
(8.95025933937286+0j)
(8.902043695656563+0j)
(8.854062701861933+0j)
(8.806316016297865+0j)
(8.758803289679737+0j)
(8.711524165088731+0j)
(8.664478283587394+0j)
(8.617665298192588+0j)
(8.571084857534334+0j)
(8.524736601283283+0j)
(8.478620160131754+0j)
(8.43273515578176+0j)
(8.387081206940444+0j)
(8.341657944189505+0j)
(8.296464992579681+0j)
(8.251501966784335+0j)
(8.206768471114152+0j)
(8.162264099539037+0j)
(8.117988435786085+0j)
(8.073941063390716+0j)
(8.030121575591615+0j)
(7.986529556444788+0j)
(7.9431645782849

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'line': {'color': 'blue', 'width': 2},
              'marker': {'size': 4},
              'mode': 'lines+markers',
              'name': '7 TeV, pl',
              'showlegend': True,
              'type': 'scatter',
              'x': [0.006, 0.007, 0.008, 0.009000000000000001,
                    0.010000000000000002, 0.011000000000000003,
                    0.012000000000000004, 0.013000000000000005,
                    0.014000000000000005, 0.015000000000000006,
                    0.016000000000000007, 0.017000000000000008,
                    0.01800000000000001, 0.01900000000000001, 0.02000000000000001,
                    0.02100000000000001, 0.022000000000000013,
                    0.023000000000000013, 0.024000000000000014,
                    0.025000000000000015, 0.026000000000000016,
                    0.027000000000000017, 0.028000000000000018,
                    0.02900000000000002, 0.03000000000000002, 0.03100000000000002,
                    0.03200000000000002, 0.03300000000000002, 0.03400000000000002,
                    0.035000000000000024, 0.036000000000000025,
                    0.037000000000000026, 0.03800000000000003, 0.03900000000000003,
                    0.04000000000000003, 0.04100000000000003, 0.04200000000000003,
                    0.04300000000000003, 0.04400000000000003, 0.04500000000000003,
                    0.046000000000000034, 0.047000000000000035,
                    0.048000000000000036, 0.04900000000000004, 0.05000000000000004,
                    0.05100000000000004, 0.05200000000000004, 0.05300000000000004,
                    0.05400000000000004, 0.05500000000000004, 0.05600000000000004,
                    0.057000000000000044, 0.058000000000000045,
                    0.059000000000000045, 0.060000000000000046,
                    0.06100000000000005, 0.06200000000000005, 0.06300000000000004,
                    0.06400000000000004, 0.06500000000000004, 0.06600000000000004,
                    0.06700000000000005, 0.06800000000000005, 0.06900000000000005,
                    0.07000000000000005, 0.07100000000000005, 0.07200000000000005,
                    0.07300000000000005, 0.07400000000000005, 0.07500000000000005,
                    0.07600000000000005, 0.07700000000000005, 0.07800000000000006,
                    0.07900000000000006, 0.08000000000000006, 0.08100000000000006,
                    0.08200000000000006, 0.08300000000000006, 0.08400000000000006,
                    0.08500000000000006, 0.08600000000000006, 0.08700000000000006,
                    0.08800000000000006, 0.08900000000000007, 0.09000000000000007,
                    0.09100000000000007, 0.09200000000000007, 0.09300000000000007,
                    0.09400000000000007, 0.09500000000000007, 0.09600000000000007,
                    0.09700000000000007, 0.09800000000000007, 0.09900000000000007,
                    0.10000000000000007, 0.10100000000000008, 0.10200000000000008,
                    0.10300000000000008, 0.10400000000000008, 0.10500000000000008,
                    0.10600000000000008, 0.10700000000000008, 0.10800000000000008,
                    0.10900000000000008, 0.11000000000000008, 0.11100000000000008,
                    0.11200000000000009, 0.11300000000000009, 0.11400000000000009,
                    0.11500000000000009, 0.11600000000000009, 0.11700000000000009,
                    0.11800000000000009, 0.11900000000000009, 0.12000000000000009,
                    0.1210000000000001, 0.1220000000000001, 0.1230000000000001,
                    0.1240000000000001, 0.12500000000000008, 0.12600000000000008,
                    0.12700000000000009, 0.12800000000000009, 0.1290000000000001,
                    0.1300000000000001, 0.1310000000000001, 0.1320000000000001,
                    0.1330000000000001, 0.1340000000000001, 0.1350000000000001,
                    0.1360000000000001, 0.1370000000000001, 0.1380000000000001,
                    0.139000000000000

In [10]:
# # PLOT BORN SIGMA TOT =============================================================
# # 
# # =============================================================


data_sigma_tot_atlas = pd.read_csv(
    "../../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

lst_born_amp = []

start_sqrt_s = 1
max_sqrt_s = 13010
step = 100

def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))

def get_sigma_tot(epsilon, mg, a1, a2, mg_model):

    lst_sigma_tot = []
    lst_sqrt_s = []

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s:

        s = sqrt_s ** 2

        integral_value = full_int(mg, a1, a2, mg_model, 0.0, sqrt_s) 

        born_amp = amp_calculation(integral_value, s, epsilon, 0)
        lst_born_amp.append(born_amp)
        
        lst_sigma_tot.append(sigma_tot(
            amp_calculation(integral_value, s, epsilon, 0), s))
        
        lst_sqrt_s.append(sqrt_s)
        sqrt_s += step
    return lst_sigma_tot, lst_sqrt_s

##-----------------------------------------------------------------------------------------------

sigma_tot_pl_atlas = get_sigma_tot(
    minuit_born.values['eps'],
    minuit_born.values['mg'],
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

sigma_tot_pl_atlas_values = sigma_tot_pl_atlas[0]
lst_sqrt_s = sigma_tot_pl_atlas[1]



fig = go.Figure()

add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

#-----------------------------------------------------------------------------------------------
#----

add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

fig.update_layout(
    title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
        range=[np.log10(2000), np.log10(14000)],
    ),
    yaxis=dict(
        title='σ_tot [mb]',
        range=[80, 125]
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig.update_xaxes(gridcolor='lightgray')
fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")

/tmp/ipykernel_19368/414226233.py:6: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'line': {'color': 'blue', 'dash': 'solid', 'width': 2},
              'marker': {'size': 3},
              'mode': 'lines+markers',
              'name': 'PL Atlas',
              'showlegend': True,
              'type': 'scatter',
              'x': [1, 101, 201, 301, 401, 501, 601, 701, 801, 901, 1001, 1101,
                    1201, 1301, 1401, 1501, 1601, 1701, 1801, 1901, 2001, 2101,
                    2201, 2301, 2401, 2501, 2601, 2701, 2801, 2901, 3001, 3101,
                    3201, 3301, 3401, 3501, 3601, 3701, 3801, 3901, 4001, 4101,
                    4201, 4301, 4401, 4501, 4601, 4701, 4801, 4901, 5001, 5101,
                    5201, 5301, 5401, 5501, 5601, 5701, 5801, 5901, 6001, 6101,
                    6201, 6301, 6401, 6501, 6601, 6701, 6801, 6901, 7001, 7101,
                    7201, 7301, 7401, 7501, 7601, 7701, 7801, 7901, 8001, 8101,
                    8201, 8301, 8401, 8501, 8601, 8701, 8801, 8901, 9001, 9101,
                    9201, 9301, 9401, 9501, 9601, 9701, 9801, 9901, 10001, 10101,
                    10201, 10301, 10401, 10501, 10601, 10701, 10801, 10901, 11001,
                    11101, 11201, 11301, 11401, 11501, 11601, 11701, 11801, 11901,
                    12001, 12101, 12201, 12301, 12401, 12501, 12601, 12701, 12801,
                    12901, 13001],
              'y': [30.532750140434267, 56.87852440989461, 61.9101549386159,
                    65.06731758383968, 67.40740433962429, 69.28156137594712,
                    70.8521285105009, 72.20815639410374, 73.40401063301627,
                    74.47542748870254, 75.44720894506449, 76.3373012907522,
                    77.15912666866903, 77.92299705225342, 78.63701351462329,
                    79.30766098116013, 79.94021470458485, 80.5390259311297,
                    81.10772755309925, 81.64938529484145, 82.16661092406754,
                    82.66164842222804, 83.13644053526748, 83.59268084865414,
                    84.0318550196073, 84.45527377609787, 84.86409958608378,
                    85.25936840492018, 85.64200755574426, 86.0128505423917,
                    86.3726494075065, 86.7220851100097, 87.06177629232046,
                    87.39228672916202, 87.71413168974296, 88.02778339879927,
                    88.333675745978, 88.63220836482057, 88.92375018032597,
                    89.20864250635915, 89.48720175999657, 89.75972184848919,
                    90.02647627527917, 90.28772000398169, 90.5436911130816,
                    90.79461226902465, 91.04069204119563, 91.2821260787923,
                    91.51909816670879, 91.75178117510565, 91.98033791530777,
                    92.20492191294316, 92.42567810778026, 92.6427434884776,
                    92.85624766940487, 93.06631341579013, 93.27305712266896,
                    93.4765892524487, 93.67701473532092, 93.8744333362582,
                    94.06893999190008, 94.26062512025551, 94.44957490582202,
                    94.63587156243604, 94.81959357591657, 95.00081592834547,
                    95.1796103056339, 95.35604528985242, 95.53018653765322,
                    95.70209694597894, 95.87183680613288, 96.03946394718295,
                    96.20503386957625, 96.3685998697582, 96.53021315651657,
                    96.68992295970334, 96.84777663192837, 97.00381974376509,
                    97.15809617296078, 97.31064818810029, 97.46151652713384,
                    97.61074047114403, 97.75835791369545, 97.90440542608198,
                    98.04891831876085, 98.19193069923804, 98.33347552665053,
                    98.47358466326867, 98.61228892312599, 98.74961811796807,
                    98.88560110069552, 99.02026580646464, 99.15363929159612,
                    99.28574777043096, 99.4166166502631, 99.54627056446792,
                    99.67473340393798, 99.8020283469295, 99.92817788741465,
                    100.05320386202952, 100.17712747570043, 100.29996932602592,
                    100.42174942648703, 100.54248722855

In [11]:
def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):
    # Garante que q_val seja array 1D
    q_val = np.atleast_1d(q_val)
    results = []

    for q in q_val:
        def integrand(y, x, mg, a1, a2, m2_func, q_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            return k * (
                T_1(k, q_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q_val, phi, mg, a1, a2, m2_func)
            ) * jacobian

        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func, q),
                0, 1, n=n_points
            )[0]

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]


In [12]:
# from scipy.integrate import quad  # kept for compatibility if used elsewhere

# lst_chi = []
# lst_amp_eik = []
# lst_diff_sigma = []

# lst_q_integration = np.linspace(0, 5, 1000)
# lst_b_integration = np.linspace(0, 15, 1000)

# # step size for Riemann sum
# dq = lst_q_integration[1] - lst_q_integration[0]

# # midpoints for improved Riemann sum accuracy
# q_midpoints = lst_q_integration[:-1] + dq / 2

# for b_val in lst_b_integration:

#     chi_sum = 0

#     for q_val in q_midpoints:

#         q2_val = q_val**2
#         sqrt_s = 7000

#         s = sqrt_s ** 2
#         t = -q2_val

#         diff_t = full_int(
#             minuit_born.values['mg'],
#             minuit_born.values['a1'],
#             minuit_born.values['a2'],
#             m2_pl,
#             q2_val,
#             sqrt_s
#         )

#         born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)

#         chi_val = (1/s) * q_val * j0(b_val * q_val) * born_amp

#         # Midpoint Riemann sum contribution
#         chi_sum += chi_val * dq

#     print(chi_sum)

#     lst_chi.append(chi_sum)


In [13]:
# import numpy as np
# from scipy.integrate import quad
# from scipy.special import j0

# lst_chi = []
# lst_amp_eik = []

# # Integration limits
# q_min, q_max = 0, 0.4
# b_min, b_max = 0, 20

# # Integration grid for outer b loop
# n_points_b = 200
# lst_b = np.linspace(b_min, b_max, n_points_b)

# sqrt_s = 13000
# s = sqrt_s ** 2


# # Define integrand in q for given b
# def chi_integrand_q(q_val, b_val):
#     q2_val = q_val ** 2
#     t = -q2_val

#     diff_t = full_int(
#         minuit_born.values['mg'],
#         minuit_born.values['a1'],
#         minuit_born.values['a2'],
#         m2_pl,
#         q2_val,
#         sqrt_s
#     )

#     born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)
#     return (1 / s) * q_val * j0(b_val * q_val) * born_amp  # complex-valued


# # Outer integration in b (explicit loop)
# def chi_integrand_b(b_val):
#     # Integrate real and imaginary parts separately
#     real_part = lambda q: np.real(chi_integrand_q(q, b_val))
#     imag_part = lambda q: np.imag(chi_integrand_q(q, b_val))

#     chi_real, _ = quad(real_part, q_min, q_max)
#     chi_imag, _ = quad(imag_part, q_min, q_max)

#     return chi_real + 1j * chi_imag


# # Perform full integration in b
# for b_val in lst_b:
#     chi_sum = chi_integrand_b(b_val)
#     print(chi_sum)
#     lst_chi.append(chi_sum)


In [14]:
# from scipy.integrate import quad
# import numpy as np
# from scipy.special import j0

# b_max = 30
# q_min, q_max = 0, 10

# lst_chi = []
# lst_b_integration = np.linspace(0, b_max, 100)

# sqrt_s = 7000
# s = sqrt_s ** 2


# # Loop over b points (so we can store chi(b) values)
# for b_val in lst_b_integration:

#     def integrand_q(q_val):
#         q2_val = q_val ** 2
#         t = -q2_val

#         diff_t = full_int(
#             minuit_born.values['mg'],
#             minuit_born.values['a1'],
#             minuit_born.values['a2'],
#             m2_pl,
#             q2_val,
#             sqrt_s
#         )
#         print(f'diff_t: {diff_t} q2_val: {q2_val}')
#         print(50*'-')

#         born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)

#         # Keep complex amplitude fully represented
#         chi_val = (1/s) * q_val * j0(b_val * q_val) * born_amp
#         return chi_val

#     # Separate integration for real and imaginary parts for numerical stability
#     # chi_real, err_real = quad(lambda q: np.real(integrand_q(q)), q_min, q_max, limit=300, epsabs=1e-8, epsrel=1e-6)
#     chi_imag, err_imag = quad(lambda q: np.imag(integrand_q(q)), q_min, q_max)

#     chi_val = 0 + 1j * chi_imag
#     lst_chi.append(chi_val)

#     # print(f"b = {b_val:.3f} -> chi_val = {chi_val}, err_imag = {err_imag}")


In [ ]:
# # from scipy.integrate import quad  # kept for compatibility if used elsewhere

# # Define lists
# lst_chi = []
# lst_b = []
# lst_amp_eik = []
# lst_diff_sigma = []

# # Integration ranges
# q_min, q_max = 0, 30
# b_min, b_max = 0, 20

# # Step sizes
# n_q = 1000
# n_b = 200
# dq = (q_max - q_min) / n_q
# db = (b_max - b_min) / n_b

# # Midpoints for q integration
# q_mid = q_min + dq / 2

# # Constants
# sqrt_s = 7000
# s = sqrt_s ** 2

# # Initialize b
# b_val = b_min

# # Loop over b using while
# while b_val <= b_max:
#     chi_sum = 0
#     q_val = q_min

#     # Inner loop over q using while
#     while q_val <= q_max - dq:
#         q_midpoint = q_val + dq / 2
#         q2_val = q_midpoint ** 2
#         t = -q2_val

#         diff_t = full_int(
#             minuit_born.values['mg'],
#             minuit_born.values['a1'],
#             minuit_born.values['a2'],
#             m2_pl,
#             q2_val,
#             sqrt_s
#         )

#         born_amp = amp_calculation(
#             diff_t,
#             s,
#             minuit_born.values['eps'],
#             t
#         )

#         chi_val = (1 / s) * q_midpoint * j0(b_val * q_midpoint) * born_amp
#         chi_sum += chi_val * dq

#         q_val += dq  # increment q

#     print(chi_sum)
#     lst_chi.append(chi_sum)

#     b_val += db  # increment b

#     # Append values to lists
#     lst_b.append(b_val)
    
import numpy as np
from scipy.special import j0

# Define lists
lst_chi = []
lst_b = []
lst_amp_eik = []
lst_diff_sigma = []

# Integration ranges
q_min, q_max = 0, 30
b_min, b_max = 0, 20

# Step sizes
n_q = 1000
n_b = 1000
dq = (q_max - q_min) / n_q
db = (b_max - b_min) / n_b

# Precompute q midpoints
q_vals = q_min + (np.arange(n_q) + 0.5) * dq
q2_vals = q_vals ** 2
t_vals = -q2_vals

# Constants
sqrt_s = 7000
s = sqrt_s ** 2

# Precompute Born amplitudes (depends only on q)
diff_t_vals = np.array([
    full_int(
        minuit_born.values['mg'],
        minuit_born.values['a1'],
        minuit_born.values['a2'],
        m2_pl,
        q2,
        sqrt_s
    )
    for q2 in q2_vals
])
born_amp_vals = np.array([
    amp_calculation(
        diff_t_vals[i],
        s,
        minuit_born.values['eps'],
        t_vals[i]
    )
    for i in range(len(q_vals))
])

# Vectorized integration over q for each b
b_vals = b_min + np.arange(n_b + 1) * db

for b_val in b_vals:
    chi_integrand = (1 / s) * q_vals * j0(b_val * q_vals) * born_amp_vals
    chi_sum = np.sum(chi_integrand) * dq  # Riemann midpoint sum
    lst_chi.append(chi_sum)
    lst_b.append(b_val)
    print(chi_sum)


12.520271643490503j
12.517143333906658j
12.507762945638204j
12.492144089630424j
12.470309415811519j
12.442290560734989j
12.408128074505246j
12.367871327211635j
12.321578395158909j
12.26931592724414j
12.211158991890347j
12.147190905006134j
12.077503039497833j
12.002194616915713j
11.921372481868952j
11.835150859894602j
11.743651099513892j
11.647001399254531j
11.54533652046021j
11.438797486747777j
11.32753127100897j
11.211690470886436j
11.091432973683416j
10.966921611692456j
10.83832380895111j
10.705811220451453j
10.56955936484539j
10.429747251699299j
10.286557004359373j
10.14017347949294j
9.990783884371616j
9.83857739295861j
9.683744761855744j
9.526477947155074j
9.366969723226168j
9.205413304452431j
9.042001970909391j
8.876928698953746j
8.710385797665081j
8.54256455205219j
8.373654873903162j
8.203844961123247j
8.033320966366412j
7.862266675726635j
7.690863198212675j
7.519288666685966j
7.347717950895319j
7.176322383194942j
7.0052694974833445j
6.834722781851031j
6.664841445374052j
6.495780

In [24]:
for i in lst_chi:
    if i < 0:
        print(i)
    

In [25]:
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.integrate import fixed_quad
# from scipy.special import j0

# # --- parâmetros já existentes ---
# # q_min, q_max, b_val, s, sqrt_s, m2_pl, minuit_born devem estar definidos
# # integrand_q já definida exatamente como no seu código anterior

# # --- valor de referência altamente preciso (fixed_quad com n=2000) ---
# chi_ref, _ = fixed_quad(integrand_q, q_min, q_max, n=2000)
# chi_ref_imag = np.imag(chi_ref)

# # --- listas de comparação ---
# n_values = [20, 50, 100, 200, 500, 1000]
# errors_riemann = []
# errors_fixed = []

# for n in n_values:
#     # Soma de Riemann
#     q_vals = np.linspace(q_min, q_max, n)
#     dq = (q_max - q_min) / (n - 1)
#     riemann_val = np.sum(integrand_q(q_vals)) * dq
#     err_r = abs((np.imag(riemann_val) - chi_ref_imag) / chi_ref_imag)
#     errors_riemann.append(err_r)

#     # Fixed Quad
#     chi_fixed, _ = fixed_quad(integrand_q, q_min, q_max, n=n)
#     err_f = abs((np.imag(chi_fixed) - chi_ref_imag) / chi_ref_imag)
#     errors_fixed.append(err_f)

# # --- gráfico de erro relativo ---
# plt.figure(figsize=(7, 5))
# plt.loglog(n_values, errors_riemann, 'o--', label='Soma de Riemann')
# plt.loglog(n_values, errors_fixed, 's-', label='Fixed Quad')
# plt.xlabel('Número de pontos de integração (n)')
# plt.ylabel('Erro relativo |Im(χ) - Im(χ_ref)| / |Im(χ_ref)|')
# plt.title('Comparação de convergência: Riemann vs Fixed Quad')
# plt.grid(True, which='both', ls='--', alpha=0.6)
# plt.legend()
# plt.show()


In [26]:
fig_chi = go.Figure()

add_total_trace(fig_chi, lst_b, np.imag(lst_chi), color='red', line_style='solid')

fig_chi.update_layout(
    title = r'$Im(\chi) \quad \text{vs.} \quad b \quad \text{(fixed sqrt(s) = 7000 GeV)}$',
    xaxis=dict(
        title='b'
    ),

    showlegend=True,
    legend=dict(
        title=r'$Im(\chi)$'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig_chi.update_xaxes(gridcolor='lightgray')
fig_chi.update_yaxes(gridcolor='lightgray')

fig_chi.show(renderer="browser")
# fig_chi.write_image('../../../../results/eikonal/b_values_plots/chi_b.pdf', width=1200, height=600)
# fig_chi.write_html('chi_sum_while.html')
